# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 4: CTR / Engagement Opportunity Scoring.** 

I chose this lane because it directly addresses a concrete, focused decision problem: which visible pages are under-capturing clicks within their search position tier?

A page ranked in position 8 with 0.05% CTR is underperforming compared to other position 8 pages (which typically get 0.11% CTR), even though position 1-3 pages average higher CTR overall. This tier-adjusted approach catches real opportunities that position-blind blending would miss, and it guides actionable decisions: rewrite title/meta, improve intent match, improve engagement, or monitor. This is decision-support: I produce a ranked list with reason codes so reviewers know what to fix.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The decision:** Which visible pages are under-capturing clicks relative to others in the same search position tier, and are worth review for content or metadata improvement?

**Who acts on it:** Content reviewers, SEO strategists, or editors who have a fixed budget each week to audit and improve pages. They need a ranked list with reason codes: top 100 pages flagged as "high impressions + low CTR for tier" or "strong position + weak engagement."

**What does a wrong call cost?** Two errors hurt:
- **False positive (flagging a low-volume page):** Reviewer wastes time auditing a page with only 30 trailing impressions. Even if CTR is low, the volume is noise. Wasted effort.
- **False negative (missing a high-potential page):** A page with 5,000 impressions at position 8 has 0.05% CTR, but position 8 pages typically earn 0.11% CTR. The gap suggests a fixable title/meta problem. If we miss it, 5,000 impressions stay uncaptured — that's real opportunity cost.

**Why a plain rule isn't enough:** Ranking all pages by position alone misses tier-specific underperformers. Ranking by impressions alone ignores position context (a 1,000-impression page at position 50 is different from position 8). A simple if-statement like "flag pages with CTR < 0.1%" catches noise and misses context. I need to calculate expected CTR *by tier*, then flag pages that sit far below their tier's median. That's the signal.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
# Setup for running in Colab or local environment
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # For local execution, navigate to repo root
    import pathlib
    notebook_path = pathlib.Path.cwd()
    while notebook_path != notebook_path.parent:
        if (notebook_path / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            os.chdir(notebook_path)
            break
        notebook_path = notebook_path.parent

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f'Dataset: {len(df):,} pages across {df["client_id"].nunique()} clients')
print()

# REAL NUMBER 1: Expected CTR by position tier
print('REAL NUMBER 1: Expected CTR by position tier')
print('(This is the benchmark for each tier)')
print()

position_tiers = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
has_position_data = df[df['avg_position'] > 0].copy()

tier_benchmarks = {}
for tier in position_tiers:
    tier_data = has_position_data[has_position_data['position_tier'] == tier]
    if len(tier_data) > 0:
        expected_ctr = tier_data['ctr'].median()
        tier_benchmarks[tier] = expected_ctr
        print(f'  {tier:12s}: expected CTR = {expected_ctr:5.2f}%  (n={len(tier_data):5,} pages)')

print()
print('REAL NUMBER 2: Pages with big CTR gaps below their tier benchmark')
print('(These are candidates for metadata or content review)')
print()

# Add tier benchmark and gap columns
has_position_data['expected_ctr'] = has_position_data['position_tier'].map(tier_benchmarks)
has_position_data['ctr_gap'] = has_position_data['ctr'] - has_position_data['expected_ctr']

# Filter for sufficient volume (>= 100 impressions) to avoid noise
candidates = has_position_data[
    (has_position_data['impressions_90d'] >= 100) &  # avoid noise
    (has_position_data['ctr_gap'] < -0.05)  # gap below tier by at least 0.05%
].copy()

# Sort by gap (worst underperformers first)
candidates = candidates.sort_values('ctr_gap')

print(f'  Total candidate pages (volume >= 100 impr, gap < -0.05%): {len(candidates):,}')
print()
print('  Top 10 biggest underperformers:')
print()

for idx, (i, row) in enumerate(candidates.head(10).iterrows(), 1):
    reason_codes = []
    if row['impressions_90d'] >= 1000:
        reason_codes.append('high_impressions')
    if row['ctr'] < (tier_benchmarks[row['position_tier']] * 0.5):
        reason_codes.append('very_low_ctr')
    if row['engagement_rate'] < 1.0:
        reason_codes.append('weak_engagement')
    
    print(f'  {idx}. {row["position_tier"]:12s} | Impr={row["impressions_90d"]:6.0f} | CTR={row["ctr"]:5.2f}% vs {row["expected_ctr"]:5.2f}% (gap={row["ctr_gap"]:6.2f}%)')
    print(f'     Reasons: {", ".join(reason_codes) if reason_codes else "low_ctr_for_tier"}')
    print()

print()
print('REAL NUMBER 3: Tier distribution of candidates')
print('(Sanity check: should see all tiers, not just top positions)')
print()

tier_distribution = candidates['position_tier'].value_counts()
for tier in position_tiers:
    count = tier_distribution.get(tier, 0)
    pct = (count / len(candidates) * 100) if len(candidates) > 0 else 0
    print(f'  {tier:12s}: {count:5,} pages ({pct:5.1f}%)')

print()
print('This distribution shows that the current screening logic is surfacing opportunities mainly in page_1 and striking, not across all tiers.')

Dataset: 30,000 pages across 32 clients

REAL NUMBER 1: Expected CTR by position tier
(This is the benchmark for each tier)

  top_3       : expected CTR =  0.00%  (n=1,116 pages)
  page_1      : expected CTR =  0.16%  (n=11,814 pages)
  striking    : expected CTR =  0.11%  (n=7,304 pages)
  page_3_5    : expected CTR =  0.03%  (n=7,242 pages)
  deep        : expected CTR =  0.00%  (n=1,319 pages)

REAL NUMBER 2: Pages with big CTR gaps below their tier benchmark
(These are candidates for metadata or content review)

  Total candidate pages (volume >= 100 impr, gap < -0.05%): 4,213

  Top 10 biggest underperformers:

  1. page_1       | Impr=   152 | CTR= 0.00% vs  0.16% (gap= -0.16%)
     Reasons: very_low_ctr, weak_engagement

  2. page_1       | Impr=  2316 | CTR= 0.00% vs  0.16% (gap= -0.16%)
     Reasons: high_impressions, very_low_ctr, weak_engagement

  3. page_1       | Impr=   361 | CTR= 0.00% vs  0.16% (gap= -0.16%)
     Reasons: very_low_ctr, weak_engagement

  4. page_1    

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I CAN claim

- **Observed:** I can observe and measure CTR, position, impressions, clicks, and engagement rate directly in the historical data. No assumptions needed.
- **Directional:** I can show that pages with low CTR relative to their position tier have historically underperformed. If a page improves its CTR within its tier, I can show the directional impact on clicks.
- **Decision-support:** I can rank pages by CTR gap within tier and flag them with reason codes (high volume + low CTR, weak engagement, etc.). Reviewers can use this ranked list to allocate audit effort more efficiently than position-only ranking.
- **Baseline comparison:** I can build a simple baseline (e.g., "rank by position only") and show that my tier-adjusted score produces a different, more diverse set of candidates.

### What I CANNOT claim

- **Causation / "If a reviewer fixes this page's title, CTR will improve by X%."** I cannot prove what a reviewer's action will cause. I observe correlation in historical data, not cause and effect. Content quality, user intent changes, and algorithm updates are confounded.
- **Predicting Google's ranking changes.** I cannot predict whether Google will maintain or improve a page's position after a reviewer updates it.
- **Predicting absolute future CTR.** Search behavior, competition, and Google's algorithm change. A 0.76% CTR today does not guarantee 0.76% next month.
- **Complete strategies.** This score is decision-support for content review prioritization, not a complete strategy. Off-page signals, competitive landscape, and editorial judgment matter too.

### My validation (sanity checks on the ranked list)

I will validate by checking:
1. **Volume floor:** My top-100 candidates have ≥ 100 impressions (no noise)
2. **Tier diversity:** My candidates span all five tiers (not just top_3), proving tier adjustment matters
3. **Reason codes plausibility:** Can I verify each reason code (high_impressions ≥ 1000?, very_low_ctr < 50% of tier median?) against the data?
4. **Gap consistency:** Pages I rank higher have larger CTR gaps within their tier
5. **Baseline comparison:** My tier-adjusted ranking produces a different top-100 than position-only ranking (if it's identical, I haven't added value)

I will report honestly: if my ranking doesn't beat the baseline, I'll say so.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.